# Task 2 - Build Time Series Forecasting Models
**Objective:** Develop, train, and evaluate ARIMA/SARIMA and LSTM models to forecast Tesla's stock price.

## 1. Imports and Setup

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.statespace.sarimax import SARIMAX
from pmdarima import auto_arima

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

plt.style.use('seaborn-v0_8-darkgrid')
pd.set_option('display.float_format', '{:.4f}'.format)
tf.random.set_seed(42)
np.random.seed(42)

## 2. Load Cleaned TSLA Data

In [ ]:
tsla = pd.read_csv('../data/processed/TSLA_cleaned.csv', index_col='Date', parse_dates=True)
tsla = tsla[['Adj Close']].copy()
tsla.columns = ['Close']
print(f"TSLA data: {tsla.shape}  |  {tsla.index.min().date()} to {tsla.index.max().date()}")
tsla.tail()

## 3. Train / Test Split (Chronological)

In [ ]:
SPLIT_DATE = '2025-01-01'

train = tsla[tsla.index < SPLIT_DATE]
test  = tsla[tsla.index >= SPLIT_DATE]

print(f"Train: {train.shape[0]} rows  ({train.index.min().date()} to {train.index.max().date()})")
print(f"Test : {test.shape[0]} rows  ({test.index.min().date()} to {test.index.max().date()})")

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(train.index, train['Close'], label='Train', color='steelblue')
ax.plot(test.index,  test['Close'],  label='Test',  color='darkorange')
ax.axvline(pd.Timestamp(SPLIT_DATE), color='red', linestyle='--', linewidth=1.5, label='Split')
ax.set_title('TSLA - Train / Test Split', fontsize=13, fontweight='bold')
ax.set_ylabel('Price (USD)')
ax.legend()
plt.tight_layout()
plt.show()

## 4. ARIMA / SARIMA Model

### 4.1 ACF / PACF Plots

In [ ]:
# Difference the series to make it stationary for ACF/PACF inspection
train_diff = train['Close'].diff().dropna()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
plot_acf(train_diff,  lags=40, ax=axes[0], title='ACF  (1st Difference)')
plot_pacf(train_diff, lags=40, ax=axes[1], title='PACF (1st Difference)', method='ywm')
plt.tight_layout()
plt.show()

### 4.2 Auto ARIMA Parameter Search

In [ ]:
print("Running auto_arima — this may take a few minutes...")
auto_model = auto_arima(
    train['Close'],
    start_p=0, start_q=0,
    max_p=3,   max_q=3,
    d=1,
    seasonal=False,
    information_criterion='aic',
    stepwise=True,
    suppress_warnings=True,
    error_action='ignore'
)
print(auto_model.summary())
best_order = auto_model.order
print(f"\nBest ARIMA order: {best_order}")

### 4.3 Fit ARIMA and Forecast

In [ ]:
arima_model = SARIMAX(
    train['Close'],
    order=best_order,
    enforce_stationarity=False,
    enforce_invertibility=False
).fit(disp=False)

print(arima_model.summary())

In [ ]:
# Forecast for the test period
n_forecast = len(test)
arima_forecast = arima_model.forecast(steps=n_forecast)
arima_forecast.index = test.index

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(train.index[-120:], train['Close'].iloc[-120:], label='Train (last 120 days)', color='steelblue')
ax.plot(test.index,  test['Close'],    label='Actual',          color='darkorange')
ax.plot(test.index,  arima_forecast,   label='ARIMA Forecast',  color='green', linestyle='--')
ax.set_title(f'ARIMA{best_order} Forecast vs Actual', fontsize=13, fontweight='bold')
ax.set_ylabel('Price (USD)')
ax.legend()
plt.tight_layout()
plt.savefig('../data/processed/arima_forecast.png', dpi=150, bbox_inches='tight')
plt.show()

### 4.4 SARIMA (Seasonal Extension)

In [ ]:
print("Running auto_arima with seasonality (m=12)...")
sarima_auto = auto_arima(
    train['Close'],
    start_p=0, start_q=0,
    max_p=2,   max_q=2,
    d=1,
    seasonal=True, m=12,
    start_P=0, start_Q=0,
    max_P=1,   max_Q=1,
    D=1,
    information_criterion='aic',
    stepwise=True,
    suppress_warnings=True,
    error_action='ignore'
)
sarima_order = sarima_auto.order
sarima_seasonal_order = sarima_auto.seasonal_order
print(f"Best SARIMA order: {sarima_order}  Seasonal: {sarima_seasonal_order}")

In [ ]:
sarima_model = SARIMAX(
    train['Close'],
    order=sarima_order,
    seasonal_order=sarima_seasonal_order,
    enforce_stationarity=False,
    enforce_invertibility=False
).fit(disp=False)

sarima_forecast = sarima_model.forecast(steps=n_forecast)
sarima_forecast.index = test.index
print("SARIMA model fitted and forecast generated.")

## 5. LSTM Model

### 5.1 Prepare Sequence Data

In [ ]:
WINDOW_SIZE = 60

scaler = MinMaxScaler(feature_range=(0, 1))
train_scaled = scaler.fit_transform(train[['Close']])
test_scaled  = scaler.transform(test[['Close']])

def create_sequences(data, window):
    X, y = [], []
    for i in range(window, len(data)):
        X.append(data[i - window:i, 0])
        y.append(data[i, 0])
    return np.array(X), np.array(y)

X_train, y_train = create_sequences(train_scaled, WINDOW_SIZE)

# For test sequences, prepend last WINDOW_SIZE train values
combined_scaled = np.concatenate([train_scaled[-WINDOW_SIZE:], test_scaled])
X_test, y_test = create_sequences(combined_scaled, WINDOW_SIZE)

# Reshape for LSTM: (samples, timesteps, features)
X_train = X_train.reshape(X_train.shape[0], X_train.shape[1], 1)
X_test  = X_test.reshape(X_test.shape[0],  X_test.shape[1],  1)

print(f"X_train: {X_train.shape}  |  y_train: {y_train.shape}")
print(f"X_test : {X_test.shape}   |  y_test : {y_test.shape}")

### 5.2 Build LSTM Architecture

In [ ]:
def build_lstm(window_size):
    model = Sequential([
        LSTM(64, return_sequences=True, input_shape=(window_size, 1)),
        Dropout(0.2),
        LSTM(32, return_sequences=False),
        Dropout(0.2),
        Dense(1)
    ])
    model.compile(optimizer='adam', loss='mean_squared_error')
    return model

lstm_model = build_lstm(WINDOW_SIZE)
lstm_model.summary()

### 5.3 Train LSTM

In [ ]:
early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

history = lstm_model.fit(
    X_train, y_train,
    epochs=100,
    batch_size=32,
    validation_split=0.1,
    callbacks=[early_stop],
    verbose=1
)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(history.history['loss'],     label='Train Loss')
ax.plot(history.history['val_loss'], label='Val Loss')
ax.set_title('LSTM Training History', fontsize=13, fontweight='bold')
ax.set_xlabel('Epoch')
ax.set_ylabel('MSE Loss')
ax.legend()
plt.tight_layout()
plt.savefig('../data/processed/lstm_training_history.png', dpi=150, bbox_inches='tight')
plt.show()

### 5.4 Generate LSTM Forecast

In [ ]:
lstm_pred_scaled = lstm_model.predict(X_test)
lstm_forecast = scaler.inverse_transform(lstm_pred_scaled).flatten()

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(train.index[-120:], train['Close'].iloc[-120:], label='Train (last 120 days)', color='steelblue')
ax.plot(test.index, test['Close'],   label='Actual',         color='darkorange')
ax.plot(test.index, lstm_forecast,   label='LSTM Forecast',  color='purple', linestyle='--')
ax.set_title('LSTM Forecast vs Actual', fontsize=13, fontweight='bold')
ax.set_ylabel('Price (USD)')
ax.legend()
plt.tight_layout()
plt.savefig('../data/processed/lstm_forecast.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Model Evaluation and Comparison

In [ ]:
def compute_metrics(actual, predicted, model_name):
    actual = np.array(actual)
    predicted = np.array(predicted)
    mae  = mean_absolute_error(actual, predicted)
    rmse = np.sqrt(mean_squared_error(actual, predicted))
    mape = np.mean(np.abs((actual - predicted) / actual)) * 100
    return {'Model': model_name, 'MAE': mae, 'RMSE': rmse, 'MAPE (%)': mape}

actual = test['Close'].values

results = [
    compute_metrics(actual, arima_forecast.values,  f'ARIMA{best_order}'),
    compute_metrics(actual, sarima_forecast.values, f'SARIMA{sarima_order}x{sarima_seasonal_order}'),
    compute_metrics(actual, lstm_forecast,           'LSTM (2-layer)')
]

results_df = pd.DataFrame(results).set_index('Model')
print("\n" + "="*55)
print("  MODEL COMPARISON TABLE")
print("="*55)
print(results_df.to_string())
print("="*55)

In [ ]:
# Combined forecast comparison plot
fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(test.index, actual,                label='Actual',          color='black',      linewidth=2)
ax.plot(test.index, arima_forecast.values, label=f'ARIMA{best_order}', color='green',  linestyle='--', linewidth=1.5)
ax.plot(test.index, sarima_forecast.values,label='SARIMA',          color='royalblue',  linestyle='-.',  linewidth=1.5)
ax.plot(test.index, lstm_forecast,         label='LSTM',            color='purple',     linestyle=':',   linewidth=2)
ax.set_title('TSLA - All Model Forecasts vs Actual (Test Period)', fontsize=13, fontweight='bold')
ax.set_ylabel('Price (USD)')
ax.set_xlabel('Date')
ax.legend()
plt.tight_layout()
plt.savefig('../data/processed/model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Metrics bar chart
fig, axes = plt.subplots(1, 3, figsize=(14, 5))
metrics_list = ['MAE', 'RMSE', 'MAPE (%)']
bar_colors = ['#2CA02C', '#1F77B4', '#9467BD']

for ax, metric in zip(axes, metrics_list):
    bars = ax.bar(results_df.index, results_df[metric], color=bar_colors, edgecolor='white', width=0.5)
    ax.set_title(metric, fontsize=12, fontweight='bold')
    ax.set_ylabel(metric)
    ax.tick_params(axis='x', rotation=15)
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() * 1.01,
                f'{bar.get_height():.2f}', ha='center', va='bottom', fontsize=9)

plt.suptitle('Model Performance Comparison', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../data/processed/metrics_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Model Selection Discussion

In [ ]:
best_model = results_df['RMSE'].idxmin()
print(f"Best model by RMSE: {best_model}")
print()
print(results_df.to_string())

### Discussion

**ARIMA/SARIMA:**
- Classical statistical models that rely on linear relationships in the differenced series.
- Fast to train and highly interpretable — parameters (p, d, q) have clear statistical meaning.
- Tend to produce forecasts that revert toward the mean over longer horizons, which limits their ability to track sharp price movements in volatile assets like TSLA.
- Best suited for short-horizon forecasts on relatively stable series.

**LSTM:**
- A recurrent deep learning architecture capable of learning long-range temporal dependencies.
- Can capture non-linear patterns and complex momentum effects that ARIMA cannot model.
- Requires more data, longer training time, and careful hyperparameter tuning.
- Typically achieves lower MAE/RMSE on volatile assets when sufficient training data is available.

**Conclusion:**
- If LSTM achieves lower RMSE/MAPE, it is the preferred model for TSLA due to its ability to model non-linear dynamics.
- In practice, both model families are used as complementary inputs — ARIMA for interpretability and LSTM for predictive power — within a broader ensemble or decision-support framework.
- Per the Efficient Market Hypothesis, neither model should be used as a standalone trading signal; they are most valuable for volatility forecasting and risk management.